# 🔬 dissectml — Quick Demo

**The missing middle layer between EDA and AutoML.**

This notebook demonstrates the full dissectml pipeline: Deep EDA → Pre-Model Intelligence → Model Battle → Interactive Report.

[![PyPI](https://img.shields.io/pypi/v/dissectml)](https://pypi.org/project/dissectml/) [![GitHub](https://img.shields.io/github/stars/rupeshbharambe24/dissectML?style=social)](https://github.com/rupeshbharambe24/dissectML)

---

## 1. Install

In [ ]:
!pip install dissectml -q

In [ ]:
import dissectml as dml
import pandas as pd
print("dissectml installed!")

## 2. Load Data

dissectml ships with built-in datasets. You can also use any pandas DataFrame.

In [ ]:
df = dml.load_titanic()
print(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

---
## 3. Stage 1 — Deep EDA

`dml.explore()` returns a lazy result — computation only happens when you access a sub-module.

In [ ]:
eda = dml.explore(df, target="survived")
eda

### 3a. Overview — Auto-detect column types and profiles

In [ ]:
print(eda.overview.summary())
eda.overview.to_dataframe()

### 3b. Correlations — Unified matrix (Pearson + Cramér's V + η for mixed types)

In [ ]:
eda.correlations.heatmap()

In [ ]:
# Top correlations table
eda.correlations.top_correlations()

### 3c. Missing Data — Counts and pattern analysis

In [ ]:
# Missing value counts per column
eda.missing.counts()

In [ ]:
# Pattern analysis (returns DataFrame of missingness patterns)
eda.missing.patterns()

In [ ]:
# Recommendations on how to handle missing data
eda.missing.recommendations()

### 3d. Outliers — Consensus detection (IQR + Z-score + Isolation Forest)

In [ ]:
# Outlier consensus across 3 methods
eda.outliers.consensus()

In [ ]:
# Outlier comparison table
eda.outliers.comparison()

### 3e. Statistical Tests — Normality, independence, group comparisons

In [ ]:
# Shapiro-Wilk normality tests for all numeric columns
eda.tests.normality()

In [ ]:
# All statistical tests at once (returns dict of test results)
all_results = eda.tests.all_tests()
print("Available test categories:", list(all_results.keys()))

# Group comparisons against the target
rows = []
for pair, res in all_results.get('group_comparisons', {}).items():
    rows.append({
        'pair': pair,
        'test': res['test_used'],
        'p_value': res['p_value'],
        'significant': res['significant']
    })
pd.DataFrame(rows)

### 3f. Cluster Discovery — K-Means with PCA visualization

In [ ]:
eda.clusters.scatter_2d()

### 3g. Target Analysis

In [ ]:
# Class balance
eda.target.balance()

---
## 4. Stage 2 — Pre-Model Intelligence

Before training any models, check if your data is ready and scan for issues.

In [ ]:
intel = dml.analyze_intelligence(df, target="survived", task="classification")

### 4a. Data Readiness Score (0-100)

In [ ]:
print(f"Readiness: {intel.readiness.score:.0f}/100 (Grade {intel.readiness.grade})")
print()
print(intel.readiness.summary())

In [ ]:
# Score breakdown by component
intel.readiness.breakdown

In [ ]:
# Visual gauge
intel.readiness.gauge_figure()

### 4b. Target Leakage Detection

In [ ]:
print(f"Leakage warnings: {len(intel.leakage)}")
for warning in intel.leakage:
    severity = warning.get('severity', 'unknown').upper()
    print(f"  [{severity}] {warning['column']}: {warning['explanation']}")

### 4c. Algorithm Recommendations

In [ ]:
# Top recommended algorithm families based on your data
print("Top 3 algorithm families:")
for i, algo in enumerate(intel.recommendations.top(3), 1):
    print(f"  {i}. {algo}")

In [ ]:
# Reasoning behind the recommendations
for reason in intel.recommendations.reasoning:
    print(f"  • {reason}")

### 4d. Multicollinearity (VIF)

In [ ]:
intel.vif

---
## 5. Stage 3 — Model Battle

Train multiple models with cross-validation in one call.

In [ ]:
# Train 6 models (use dml.battle(df, target='survived') for all 19)
models = dml.battle(
    df, 
    target="survived",
    models=[
        "LogisticRegression",
        "RandomForestClassifier",
        "GradientBoostingClassifier",
        "GaussianNB",
        "KNeighborsClassifier",
        "DecisionTreeClassifier",
    ]
)

print(f"Best: {models.best.name} (accuracy={models.best.metrics['accuracy']:.4f})")
print()
models.leaderboard()

---
## 6. Full Pipeline — One Function Call

Run all stages together with `dml.analyze()`.

In [ ]:
report = dml.analyze(
    df, 
    target="survived",
    battle_models=["LogisticRegression", "RandomForestClassifier", "GradientBoostingClassifier"]
)

print(report.summary())

### Export Interactive HTML Report

In [ ]:
path = report.export("dissectml_report.html")
print(f"Report saved to: {path}")

# In Colab, download the file
try:
    from google.colab import files
    files.download("dissectml_report.html")
except ImportError:
    print("Open the file in your browser to view the interactive report.")

---
## 7. Use Your Own Data

Replace the Titanic dataset with any pandas DataFrame:

```python
import pandas as pd
df = pd.read_csv("your_data.csv")
report = dml.analyze(df, target="your_target_column")
report.export("my_report.html")
```

---

## Links

- **GitHub:** [github.com/rupeshbharambe24/dissectML](https://github.com/rupeshbharambe24/dissectML)
- **PyPI:** [pypi.org/project/dissectml](https://pypi.org/project/dissectml/)
- **Install:** `pip install dissectml`

If this was useful, please ⭐ the repo on GitHub — it helps with discoverability!

Found a bug? [Open an issue](https://github.com/rupeshbharambe24/dissectML/issues) — this is a v0.1.2 first-release library and feedback is welcomed.